# leodragkit aero tutorial: building and plotting a spacecraft

A running example that grows with each aero task: every task adds a section showing what its code produces. Tests check the numbers; this notebook shows the result.

**Before running.** Activate the `leodragkit` environment and install the optional plotting library: `conda install -c conda-forge plotly`. The figures are interactive: drag to rotate, scroll to zoom, click a legend entry to hide it.

**In git.** Clear all outputs before committing (in Jupyter or VS Code: *Clear All Outputs*), so the repository holds code and text, not large figures.

| Section | Task | What it shows |
| --- | --- | --- |
| 1 | A3a | one triangle: corner order sets the outward normal |
| 2 | A3a | a cube built by hand as a `PanelSet` |
| 3 | A3p | the cube in 3-D, coloured by component and by material |
| 4 | A3a, A3p | a macro model: panels without positions |
| 5 | A3b1 | primitives (box, plate, cylinder) at two resolutions, placed by poses |
| 6 | A3b2 | a 3U CubeSat joined from components |
| 7 | A3b3 | orientation checks: the summary table, a broken box, a warning |
| 8 | A3c | placing parts: planes and rotations |
| 9 | A3c | a spacecraft from a YAML file, with its materials |
| 10 | A3d1 | a spacecraft from a mesh file (STL), on its own and in a description |
| 11 | A3d2 | a macro model: a table of panels, with and without centres; GRACE's published panels |
| 12 | A4a | attitude in ITRF: Spire files (or the attitude law) and the direction the air comes from |
| 13 | A4b | the flow on each panel: cos δ on the 3U example under section 12's attitude, and the area facing the flow |


In [1]:
# %load_ext autoreload
# %autoreload 2

import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [2]:
import numpy as np

from leodragkit.aero.geometry import (
    compute_triangle_properties,
    panel_set_from_panels,
    panel_set_from_triangles,
)
from leodragkit.plots.spacecraft import plot_panel_set_3d

## 1. One triangle (A3a)

A triangle's outward normal follows its corner order: counter-clockwise when seen from outside. These corners go counter-clockwise seen from +z, so the normal points along +z. The area is half the length of the cross product of two edges.

In [3]:
triangle_m = np.array([[[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]])
properties = compute_triangle_properties(triangle_m)
print("normal:  ", properties.outward_normals[0])
print("area:    ", properties.areas_m2[0], "m²")
print("centroid:", properties.centroids_m[0], "m")

normal:   [0. 0. 1.]
area:     0.5 m²
centroid: [0.33333333 0.33333333 0.        ] m


Listing the same corners in the opposite order flips the normal: the panel now faces −z.

In [4]:
reversed_properties = compute_triangle_properties(triangle_m[:, ::-1, :])
print("normal with the corners reversed:", reversed_properties.outward_normals[0])

normal with the corners reversed: [ 0.  0. -1.]


## 2. A cube built by hand (A3a)

A unit cube centred on the body origin: six faces, two triangles each, every face counter-clockwise seen from outside. Task A3b will replace this by-hand construction with primitives (`box`, `plate`, `cylinder`); writing it out once shows what those primitives produce.

The four side faces are the component `body` and the two z faces are `end_caps`; the ±y faces are `solar_cell` material and the rest `aluminium`.

In [5]:
corners_m = np.array([[x, y, z] for x in (-0.5, 0.5) for y in (-0.5, 0.5) for z in (-0.5, 0.5)])
faces = [
    (0, 1, 3, 2),
    (4, 6, 7, 5),
    (0, 4, 5, 1),
    (2, 3, 7, 6),
    (0, 2, 6, 4),
    (1, 5, 7, 3),
]  # −x +x −y +y −z +z
cube_triangles_m = np.array(
    [corners_m[list(triangle)] for a, b, c, d in faces for triangle in ((a, b, c), (a, c, d))]
)

cube = panel_set_from_triangles(
    cube_triangles_m,
    component_ids=[0] * 8 + [1] * 4,
    material_ids=[0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0],
    component_names=("body", "end_caps"),
    material_names=("aluminium", "solar_cell"),
    source="unit cube, built by hand",
)
print(cube.panel_count, "panels; total area", cube.areas_m2.sum(), "m²")
print("has positions:", cube.has_positions, "| has triangles:", cube.has_triangles)

12 panels; total area 6.0 m²
has positions: True | has triangles: True


## 3. The cube in 3-D (A3p)

Each component is one trace, so the legend lists them. The lines are the outward normals, one per triangle, starting at its centroid; the red, green and blue lines are the body x, y and z axes. Every normal should point away from the cube: one pointing inward would mean a triangle with its corners in the wrong order.

In [6]:
figure = plot_panel_set_3d(cube)
figure

The same cube coloured by material: the ±y faces are solar cells.

In [7]:
plot_panel_set_3d(cube, color_by="material")

## 4. A macro model: panels without positions (A3a, A3p)

Published macro models list each panel's outward normal, area and material, often with no position. Here is a simple one for a 3U CubeSat (0.1 × 0.1 × 0.34 m) with one deployed solar panel (0.3 × 0.1 m), listed once for each side. The normals are rounded to four decimals, as tables often are; `panel_set_from_panels` normalises them.

With no positions there is no shape to draw. The plot shows each normal from the origin, with length proportional to the panel's area, and the title says so. A set like this supports forces, but not shadowing or moments.

In [8]:
normals = [
    [1, 0, 0],
    [-1, 0, 0],
    [0, 1, 0],
    [0, -1, 0],
    [0, 0, 1],
    [0, 0, -1],  # bus faces
    [0.7071, 0.7071, 0],
    [-0.7071, -0.7071, 0],  # one deployed panel, front and back, rounded
]
areas_m2 = [0.034, 0.034, 0.034, 0.034, 0.01, 0.01, 0.03, 0.03]
macro_model = panel_set_from_panels(
    normals,
    areas_m2,
    component_ids=[0] * 6 + [1] * 2,
    material_ids=[0] * 6 + [1] * 2,
    component_names=("bus", "deployed_panel"),
    material_names=("aluminium", "solar_cell"),
    source="3U CubeSat macro model (illustrative)",
)
plot_panel_set_3d(macro_model)

## 5. Primitives and poses (A3b1)

Plates, boxes and cylinders are built centred on their own origin, then placed in the body frame by a pose: a centre plus the component's own x and z axes (y = z × x, so the axes are always right-handed). `max_edge_length_m` sets the resolution; `None` gives the coarsest mesh. The triangle edges are drawn so the resolution is visible.

Until joining arrives in A3b2, a small helper turns one primitive's triangles into a `PanelSet` with a single component and material.

In [9]:
from leodragkit.aero.geometry_builder import (
    build_box_triangles,
    build_cylinder_triangles,
    build_plate_triangles,
    make_pose,
    place_triangles,
)

In [10]:
def single_component(triangles_m, name, material="aluminium"):
    """A PanelSet of one component and one material, for looking at a single primitive."""
    count = len(triangles_m)
    return panel_set_from_triangles(triangles_m, [0] * count, [0] * count, (name,), (material,), source=name)

A 3U CubeSat bus (0.1 × 0.1 × 0.34 m) at the coarsest resolution, then with no triangle edge longer than 2 cm. The force on the box is the same either way; the finer mesh only matters for shadowing.

In [11]:
bus_coarse = single_component(build_box_triangles(0.1, 0.1, 0.34), "bus, coarsest")
bus_fine = single_component(build_box_triangles(0.1, 0.1, 0.34, max_edge_length_m=0.02), "bus, 2 cm")
print(bus_coarse.panel_count, "triangles coarse;", bus_fine.panel_count, "triangles at 2 cm")
plot_panel_set_3d(bus_fine)

12 triangles coarse; 780 triangles at 2 cm


In [12]:
import sys

import numpy
import plotly

print(sys.executable, numpy.__version__, plotly.__version__)

c:\Users\murph\miniconda3\envs\leodragkit\python.exe 2.2.6 7.1.0


A deployed solar panel (0.3 × 0.1 m, both sides), placed beside the bus and tilted by 45° about the body x axis: its own x stays along body x, and its normal (own z) turns from +z towards +y. The axes are typed rounded, as they would be in a YAML file; `make_pose` makes them exact.

In [13]:
panel_pose = make_pose(centre_m=[0.0, 0.2, 0.0], x_axis=[1, 0, 0], z_axis=[0, 0.7071, 0.7071])
panel_m = place_triangles(build_plate_triangles(0.3, 0.1, max_edge_length_m=0.05, two_sided=True), panel_pose)
plot_panel_set_3d(single_component(panel_m, "deployed panel", "solar_cell"), color_by="material")

A capped cylinder, such as a boom or an antenna mast: the side is faceted into segments (at least eight), and the caps are fans from their centres. Every normal should point away from the axis or along it.

In [14]:
boom_m = build_cylinder_triangles(0.02, 0.3, max_edge_length_m=0.02)
plot_panel_set_3d(single_component(boom_m, "boom"))

## 6. A spacecraft from components (A3b2)

A `Component` is one named part with one material, already placed in the body frame. `join_components` turns a list of them into one `PanelSet`: every triangle keeps its component and material, components of the same material share a material id, and the result is validated like any other `PanelSet`.

Here is a 3U CubeSat: an aluminium bus, long axis along z, with two solar panels deployed sideways from its top end, both facing +x and exposed on both sides, and a thin antenna along −z.

In [15]:
from leodragkit.aero.geometry_builder import Component, join_components

resolution_m = 0.02
bus = Component("bus", "aluminium", build_box_triangles(0.1, 0.1, 0.34, max_edge_length_m=resolution_m))


def deployed_panel(name, side):
    """A 0.3 × 0.1 m two-sided panel beside the bus on the ±y side (side = +1 or −1), facing +x."""
    pose = make_pose(centre_m=[0.0, side * (0.05 + 0.15), 0.12], x_axis=[0, side, 0], z_axis=[1, 0, 0])
    plate = build_plate_triangles(0.3, 0.1, max_edge_length_m=resolution_m, two_sided=True)
    return Component(name, "solar_cell", place_triangles(plate, pose))


antenna_pose = make_pose(centre_m=[0.0, 0.0, -0.27], x_axis=[1, 0, 0], z_axis=[0, 0, 1])
antenna = Component(
    "antenna", "aluminium", place_triangles(build_cylinder_triangles(0.005, 0.2, capped=False), antenna_pose)
)

cubesat = join_components(
    [bus, deployed_panel("panel_plus_y", +1), deployed_panel("panel_minus_y", -1), antenna],
    source="3U CubeSat with two deployed panels (tutorial)",
)
print(cubesat.panel_count, "triangles")
print("components:", cubesat.component_names)
print("materials: ", cubesat.material_names)

1396 triangles
components: ('bus', 'panel_plus_y', 'panel_minus_y', 'antenna')
materials:  ('aluminium', 'solar_cell')


Coloured by component, each part can be hidden from the legend; coloured by material, the two panels share one colour. The panels face +x on one side and −x on the other: as two-sided plates, both faces are exposed.

In [16]:
plot_panel_set_3d(cubesat)

In [17]:
plot_panel_set_3d(cubesat, color_by="material", show_edges=False)

## 7. Checking that normals point outward (A3b3)

A triangle listed clockwise has a normal pointing into the spacecraft, which silently turns a ram face into a wake face. `join_components` now checks every component as it joins them:

- a **closed** component (a box, a capped cylinder) must be consistently wound with a positive enclosed volume, or joining raises an error;
- a **two-sided** open component (a deployed panel) faces both ways by construction;
- a **one-sided** open component is flagged with a `GeometryOrientationWarning` when most of its area faces the spacecraft's centre.

`summarise_geometry` shows the result per component; in a notebook it displays as a table, with rows needing attention shaded.

In [18]:
from leodragkit.aero.geometry_checks import check_component, summarise_geometry

summarise_geometry(cubesat)

component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,780,0.156,yes,yes,no,0.0034,"ok: closed, outward"
panel_plus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
panel_minus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
antenna,16,0.006123,no,yes,no,–,"ok: open, one-sided"


A box with one triangle listed in the wrong order is caught, and the message names the triangles on either side of the broken edge:

In [19]:
broken_box_m = build_box_triangles(0.2, 0.2, 0.2)
broken_box_m[0] = broken_box_m[0][::-1]  # reverse one triangle's corners

try:
    check_component("broken box", broken_box_m)
except ValueError as error:
    print(error)

Component 'broken box': triangles 0 and 1 run along a shared edge in the same direction, so one of them is flipped: listed clockwise, not counter-clockwise, seen from outside.


Plotted with its normals, the flipped triangle's normal points into the box. (`single_component` builds a `PanelSet` directly, without the checks, so the broken box can be drawn.)

In [20]:
plot_panel_set_3d(single_component(broken_box_m, "broken box"), normal_length_m=0.08)

A one-sided panel typed facing the bus instead of away from it can't be proven wrong (it could be deliberate), so it is a warning rather than an error. The warning names the component and its panels; the summary row is shaded.

In [21]:
facing_bus_pose = make_pose(centre_m=[0.0, 0.25, 0.0], x_axis=[1, 0, 0], z_axis=[0, -1, 0])
one_sided_panel = Component(
    "panel_facing_bus", "solar_cell", place_triangles(build_plate_triangles(0.3, 0.1), facing_bus_pose)
)
with_warning = join_components([bus, one_sided_panel], source="bus with a one-sided panel facing it")
summarise_geometry(with_warning)

C:\Users\murph\AppData\Local\Temp\ipykernel_43876\978508741.py:5: GeometryOrientationWarning: bus with a one-sided panel facing it: component 'panel_facing_bus' is one-sided and 2 of its panels face the spacecraft's centre (panels 780, 781). If it should face outward, turn its pose's z axis round; if it is exposed on both sides, build it two-sided.
  with_warning = join_components([bus, one_sided_panel], source="bus with a one-sided panel facing it")


component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,780,0.156,yes,yes,no,0.0034,"ok: closed, outward"
panel_facing_bus,2,0.03,no,yes,no,–,check: 2 panel(s) face the body


To make such warnings errors, for example in a production run:

```python
import warnings
warnings.simplefilter("error", GeometryOrientationWarning)
```

## 8. Placing parts: planes and rotations (A3c)

A pose fixes where a part sits (`centre_m`) and which way it faces. There are two ways to give the orientation:

- **Axes:** `z_axis` is where the part's own +z (a plate's front face) points, and `x_axis` is where its own x (a plate's first side) runs. They must be perpendicular; the third axis is y = z × x.
- **Turn:** turn by `rotation_angle_deg` about `rotation_axis`, an axis of the part's own frame, by the right-hand rule: with your right thumb along the axis, a positive angle turns the way your fingers curl. So +30° about +x turns +z towards **−y**. Without axes the part starts unrotated, so its own axes are the body axes.
- **Both:** place by axes, then turn about the part's own axis, for example tilting a panel about its own long side.

Below, the same 0.3 × 0.1 m one-sided plate is placed eight ways, side by side along body x. Each line from a plate is its outward normal. The guide `docs/guides/describing_a_spacecraft.md` explains each case.

In [22]:
from leodragkit.aero.geometry_builder import make_pose_from_rotation, turn_pose_about_own_axis

placements = {  # name: pose
    "xy plane, facing +z": make_pose([0.0, 0, 0], x_axis=[1, 0, 0], z_axis=[0, 0, 1]),
    "xz plane, facing +y": make_pose([0.6, 0, 0], x_axis=[1, 0, 0], z_axis=[0, 1, 0]),
    "yz plane, facing +x": make_pose([1.2, 0, 0], x_axis=[0, 1, 0], z_axis=[1, 0, 0]),
    "yz plane, facing -x": make_pose([1.8, 0, 0], x_axis=[0, 1, 0], z_axis=[-1, 0, 0]),
    "+30 deg about x (rotation form)": make_pose_from_rotation(
        [2.4, 0, 0], rotation_axis=[1, 0, 0], rotation_angle_deg=30
    ),
    "-30 deg about x (axes form)": make_pose([3.0, 0, 0], x_axis=[1, 0, 0], z_axis=[0, 0.5, 0.866]),
    "facing (1, 1, 1)": make_pose([3.6, 0, 0], x_axis=[0.7071, -0.7071, 0], z_axis=[0.5774, 0.5774, 0.5774]),
    "facing +x, then 20 deg about own x": turn_pose_about_own_axis(
        make_pose([4.2, 0, 0], x_axis=[0, 1, 0], z_axis=[1, 0, 0]),
        rotation_axis=[1, 0, 0],
        rotation_angle_deg=20,
    ),
}
for name, pose in placements.items():
    print(f"{name:34s} normal (z axis) = {np.round(pose.z_axis, 3)}   own x along {np.round(pose.x_axis, 3)}")

xy plane, facing +z                normal (z axis) = [0. 0. 1.]   own x along [1. 0. 0.]
xz plane, facing +y                normal (z axis) = [0. 1. 0.]   own x along [1. 0. 0.]
yz plane, facing +x                normal (z axis) = [1. 0. 0.]   own x along [0. 1. 0.]
yz plane, facing -x                normal (z axis) = [-1.  0.  0.]   own x along [0. 1. 0.]
+30 deg about x (rotation form)    normal (z axis) = [ 0.    -0.5    0.866]   own x along [1. 0. 0.]
-30 deg about x (axes form)        normal (z axis) = [0.    0.5   0.866]   own x along [1. 0. 0.]
facing (1, 1, 1)                   normal (z axis) = [0.577 0.577 0.577]   own x along [ 0.707 -0.707  0.   ]
facing +x, then 20 deg about own x normal (z axis) = [ 0.94   0.    -0.342]   own x along [0. 1. 0.]


To draw them together without the orientation checks (several one-sided plates facing every which way would rightly be flagged), the triangles go straight into one `PanelSet`, one component per placement:

In [23]:
plate_m = build_plate_triangles(0.3, 0.1)
gallery_triangles = np.concatenate([place_triangles(plate_m, pose) for pose in placements.values()])
gallery = panel_set_from_triangles(
    gallery_triangles,
    component_ids=np.repeat(np.arange(len(placements)), len(plate_m)),
    material_ids=[0] * len(gallery_triangles),
    component_names=tuple(placements),
    material_names=("solar_cell",),
    source="placement gallery",
)
pfig = plot_panel_set_3d(gallery, normal_length_m=0.2)
pfig.update_traces(visible="legendonly", selector=dict(name="body y"))
pfig.update_traces(visible="legendonly", selector=dict(name="body z"))
pfig.show()

## 9. A spacecraft from a YAML file (A3c)

Most spacecraft are described in a short YAML file rather than in Python. `read_spacecraft_description_file` reads it (from `leodragkit.io`), and `build_spacecraft` checks every key, builds each component at its pose, joins them (running the orientation checks) and attaches the materials. This is the same 3U CubeSat as section 6, from `config/geometry/cubesat_3u_example.yaml`.

In [24]:
from pathlib import Path

from leodragkit.aero.spacecraft import build_spacecraft
from leodragkit.io.geometry_files import read_spacecraft_description_file

example_path = Path("..") / "config" / "geometry" / "cubesat_3u_example.yaml"
description = read_spacecraft_description_file(example_path)
cubesat_from_file = build_spacecraft(description, source=example_path.name)
print(
    cubesat_from_file.name,
    "|",
    cubesat_from_file.mass_kg,
    "kg |",
    cubesat_from_file.panel_set.panel_count,
    "triangles",
)
for material in cubesat_from_file.materials:
    wall, alpha = material.wall_temperature_kelvin, material.energy_accommodation
    print(f"  {material.name}: wall {wall} K, accommodation {alpha}")

3U CubeSat with deployed panels (example) | 4.0 kg | 1700 triangles
  aluminium: wall 300.0 K, accommodation 0.9
  solar_cell: wall 300.0 K, accommodation 0.9


In [25]:
plot_panel_set_3d(cubesat_from_file.panel_set)

In [26]:
summarise_geometry(cubesat_from_file.panel_set)

component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,780,0.156,yes,yes,no,0.0034,"ok: closed, outward"
panel_plus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
panel_minus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
antenna,320,0.006123,no,yes,no,–,"ok: open, one-sided"


A mistake in the file is reported with its place in the description. Here `size_m` is mistyped as `sizes_m` in the second component:

In [27]:
import copy

mistyped = copy.deepcopy(description)
mistyped["components"][1]["sizes_m"] = mistyped["components"][1].pop("size_m")
try:
    build_spacecraft(mistyped, source="mistyped example")
except ValueError as error:
    print(error)

components[1] (panel_plus_y) is missing ['size_m'] and has unknown keys ['sizes_m']; allowed: ['material', 'max_edge_length_m', 'name', 'pose', 'shape', 'size_m', 'two_sided'].


A material defined but used by no component is left out, with a warning in case it was meant to be used:

In [28]:
with_spare_material = copy.deepcopy(description)
with_spare_material["materials"]["gold_foil"] = {
    "wall_temperature_kelvin": 300.0,
    "energy_accommodation": 0.8,
}
_ = build_spacecraft(with_spare_material, source="example with a spare material")

C:\Users\murph\AppData\Local\Temp\ipykernel_43876\983870762.py:6: SpacecraftDescriptionWarning: example with a spare material: materials ['gold_foil'] are defined but used by no panel, so they are left out. If a panel should use one, check its material's spelling.
  _ = build_spacecraft(with_spare_material, source="example with a spare material")


## 10. A spacecraft from a mesh file (A3d1)

CAD tools export shapes as STL or OBJ files. `leodragkit.io` reads both (`read_stl_file`, `read_obj_file`), converting the coordinates to metres from the file's `length_unit`, which is required because CAD files are often in millimetres.

There is no CAD file in the repository, so this section first writes one: a cylindrical bus, 0.6 m across and 0.8 m tall, saved as a binary STL file **in millimetres** in a temporary folder. A binary STL file is an 80-byte header, a triangle count, and 50 bytes per triangle (a normal, three corners, two spare bytes).

In [29]:
import tempfile

from leodragkit.io.mesh_files import BINARY_STL_RECORD, read_stl_file

mesh_folder = Path(tempfile.mkdtemp())
bus_m = build_cylinder_triangles(0.3, 0.8, max_edge_length_m=0.1)  # a capped cylinder, in metres

records = np.zeros(len(bus_m), dtype=BINARY_STL_RECORD)
records["vertices"] = bus_m * 1000.0  # written in millimetres, as many CAD exports are
records["normal"] = compute_triangle_properties(bus_m).outward_normals
header = b"leodragkit tutorial".ljust(80, b" ") + np.uint32(len(bus_m)).tobytes()
(mesh_folder / "bus.stl").write_bytes(header + records.tobytes())
print(f"wrote {len(bus_m)} triangles to {mesh_folder / 'bus.stl'}")

wrote 342 triangles to C:\Users\murph\AppData\Local\Temp\tmpwgfivpdi\bus.stl


**Read it directly.** Two lines give a one-part spacecraft, checked for outward normals like any other:

In [30]:
from leodragkit.aero.materials import SurfaceMaterial
from leodragkit.aero.spacecraft import build_spacecraft_from_triangles

mesh = read_stl_file(mesh_folder / "bus.stl", length_unit="mm")
aluminium = SurfaceMaterial("aluminium", wall_temperature_kelvin=300.0, energy_accommodation=0.9)
bus_only = build_spacecraft_from_triangles(
    mesh.triangle_vertices_m, name="bus", material=aluminium, mass_kg=150.0, source=mesh.source
)
print(
    mesh.source,
    "|",
    bus_only.panel_set.panel_count,
    "triangles, largest extent",
    np.ptp(mesh.triangle_vertices_m.reshape(-1, 3), axis=0),
    "m",
)
plot_panel_set_3d(bus_only.panel_set, normal_length_m=0.1)

bus.stl (mm) | 342 triangles, largest extent [0.59590839 0.59795068 0.8       ] m


**Or combine it with other parts in a description.** A component with `shape: mesh` names its file (relative to the description), its unit, and optionally `reverse_winding`. `load_spacecraft_description` reads the YAML and every mesh it names, so the meshed bus and two hand-described panels become one spacecraft:

In [31]:
from leodragkit.io.geometry_files import load_spacecraft_description

(mesh_folder / "meshed_spacecraft.yaml").write_text(
    """
name: meshed bus with two panels (tutorial)
mass_kg: 160.0
max_edge_length_m: 0.1
materials:
  aluminium: {wall_temperature_kelvin: 300.0, energy_accommodation: 0.9}
  solar_cell: {wall_temperature_kelvin: 300.0, energy_accommodation: 0.9}
components:
  - name: bus
    shape: mesh
    file: bus.stl
    length_unit: mm
    material: aluminium
    pose: {centre_m: [0, 0, 0], x_axis: [1, 0, 0], z_axis: [0, 0, 1]}
  - name: panel_plus_y
    shape: plate
    size_m: [1.0, 0.6]
    two_sided: true
    material: solar_cell
    pose: {centre_m: [0, 0.8, 0], x_axis: [0, 1, 0], z_axis: [1, 0, 0]}
  - name: panel_minus_y
    shape: plate
    size_m: [1.0, 0.6]
    two_sided: true
    material: solar_cell
    pose: {centre_m: [0, -0.8, 0], x_axis: [0, -1, 0], z_axis: [1, 0, 0]}
""",
    encoding="utf-8",
)


meshed = build_spacecraft(
    load_spacecraft_description(mesh_folder / "meshed_spacecraft.yaml"), source="meshed_spacecraft.yaml"
)
plot_panel_set_3d(meshed.panel_set)

In [32]:
summarise_geometry(meshed.panel_set)

component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,342,2.056339,yes,yes,no,0.222094,"ok: closed, outward"
panel_plus_y,240,1.2,no,–,yes,–,"ok: open, two-sided"
panel_minus_y,240,1.2,no,–,yes,–,"ok: open, two-sided"


## 11. A macro model: a table of panels (A3d2)

A macro model lists each flat panel's normal, area, material and (optionally) centre, without its shape. leodragkit doesn't invent triangles from it, so there is no shadowing and the orientation checks can't run; it is enough for a **convex** spacecraft without deployables, and for reproducing published results (guide, section 5b).

Here is the 3U CubeSat as a macro model, `config/geometry/cubesat_3u_macro_model.yaml` with its table. The deployed panels are two rows each, front and back. With centres given, each normal is drawn from its panel's centre:

In [33]:
macro_path = Path("..") / "config" / "geometry" / "cubesat_3u_macro_model.yaml"
macro_cubesat = build_spacecraft(load_spacecraft_description(macro_path), source=macro_path.name)
print(
    macro_cubesat.panel_set.panel_count,
    "panels | triangles:",
    macro_cubesat.panel_set.has_triangles,
    "| positions:",
    macro_cubesat.panel_set.has_positions,
)
plot_panel_set_3d(macro_cubesat.panel_set, normal_length_m=0.1)

10 panels | triangles: False | positions: True


In [34]:
summarise_geometry(macro_cubesat.panel_set)

component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,6,0.156,–,–,–,–,not checkable (no triangles)
panel_plus_y,2,0.06,–,–,–,–,not checkable (no triangles)
panel_minus_y,2,0.06,–,–,–,–,not checkable (no triangles)


Many published tables give no centres. The same table without its centroid columns, read directly with `read_macro_model_file` and built with `build_spacecraft_from_macro_model`: there is nowhere to draw the panels, so each normal starts at the origin with its length proportional to the panel's area, and the title says so.

In [35]:
import csv

from leodragkit.aero.spacecraft import build_spacecraft_from_macro_model
from leodragkit.io.geometry_files import read_macro_model_file

with open(macro_path.with_suffix(".csv"), encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))
kept = ["component", "material", "normal_x", "normal_y", "normal_z", "area_m2"]
with open(mesh_folder / "no_centroids.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=kept, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

table = read_macro_model_file(mesh_folder / "no_centroids.csv")
no_positions = build_spacecraft_from_macro_model(
    table,
    name="3U CubeSat, no centres",
    materials=dict(zip(macro_cubesat.panel_set.material_names, macro_cubesat.materials)),
    mass_kg=4.0,
    source=table.source,
)
plot_panel_set_3d(no_positions.panel_set)

A real one: **GRACE**, whose eight flat surfaces are published in the GRACE Product Specification Document (Bettadpur, 2012, GRACE 327-720, §I.5.3.5), in the GRACE Satellite Frame: x towards the K-band horn, z towards nadir in flight. The large sloped panels are the solar arrays on the two sides (outer), each with a small underside (inner). No positions are given, so the arrows start at the origin, scaled by area: the nadir face is the largest, the front and rear the smallest. The table's provenance, and what isn't from the source, is written at the top of `config/geometry/grace_macro_model.yaml`.

In [36]:
grace_path = Path("..") / "config" / "geometry" / "grace_macro_model.yaml"
grace = build_spacecraft(load_spacecraft_description(grace_path), source=grace_path.name)
print(
    grace.name,
    "|",
    grace.mass_kg,
    "kg |",
    grace.panel_set.panel_count,
    "panels, total area",
    round(grace.panel_set.areas_m2.sum(), 4),
    "m²",
)
plot_panel_set_3d(grace.panel_set)

GRACE (macro model) | 487.2 kg | 8 panels, total area 16.9163 m²


## 12. Attitude in ITRF: which way the air comes from (A4a)

An `Attitude` holds `body_from_reference`, the rotation that takes a vector in a reference frame into the body's own axes. Spire's attitude files give it relative to Spire's **orbit level frame**: Z to nadir, Y opposite the orbit's angular momentum, X along the flight direction, built from the position and the *inertial* velocity (D-85). `attitude_in_itrf` turns that into ITRF using the orbit at the same instants, two hops: ITRF → orbit frame → body.

Then the velocity relative to the air, which is simply the ITRF velocity (the atmosphere co-rotates, D-25), can be written in the body frame. The air arrives **from** that direction.

**With your own Spire files:** set the directory below. Every leoOrb and leoAtt file there for the satellite is read with `io`, and each kind is merged across files (where files overlap, orbits keep the most central value, attitude the observed one). Then only observed attitude epochs are kept, matched second for second with `common_epochs` (nothing is interpolated), and turned into ITRF. **Without them,** a circular orbit with the attitude law (+x ahead, +z down) is used instead.

In [ ]:
from leodragkit.aero.frames import (
    OrbitStates,
    attitude_from_law,
    attitude_from_quaternions,
    attitude_in_itrf,
    common_epochs,
)

# Set these to your files: every matching file in the directory is read and merged.
SPIRE_DIRECTORY = Path("D:/data/spire")
SATELLITE = "FM134"
SP3_PATHS = sorted(SPIRE_DIRECTORY.glob(f"spire_nav_L1B_leoOrb_*_{SATELLITE}.sp3"))
ATTITUDE_PATHS = sorted(SPIRE_DIRECTORY.glob(f"spire_att_L1A_leoAtt_*_{SATELLITE}.log"))
use_spire_files = bool(SP3_PATHS) and bool(ATTITUDE_PATHS)
if use_spire_files:
    print(f"Using {len(SP3_PATHS)} orbit and {len(ATTITUDE_PATHS)} attitude files for {SATELLITE}.")
else:
    print("Spire files not found: using a circular orbit and the attitude law.")

In [ ]:
import warnings

if use_spire_files:
    from leodragkit.config import load_main_config
    from leodragkit.io.merging import read_sp3_files, read_spire_attitude_files
    from leodragkit.io.records import keep_observed, source_names
    from leodragkit.io.spice_kernels import load_leap_second_kernel

    kernel = load_leap_second_kernel(load_main_config().data_directories.spice_kernels, allow_download=True)
    with warnings.catch_warnings(record=True) as shown:  # merging and keep_observed say what they did
        warnings.simplefilter("always")
        sp3 = read_sp3_files(SP3_PATHS, kernel)
        records = keep_observed(read_spire_attitude_files(ATTITUDE_PATHS, kernel))
    for warning in shown:
        print(warning.message)

    # aero's own types, on GPS times (aero compares times; it never converts them).
    states = OrbitStates(
        sp3.epochs.gps, "GPS", sp3.positions_m, sp3.velocities_m_per_s, sp3.frame, source_names(sp3)
    )
    attitude = attitude_from_quaternions(
        records.epochs.gps,
        "GPS",
        records.quaternions,
        scalar_position=records.scalar_position,
        algebra=records.algebra,
        direction=records.direction,
        frame=records.frame,
        source=source_names(records),
    )
    attitude, states = common_epochs(attitude, states)
    attitude = attitude_in_itrf(attitude, states)
else:
    # A circular polar orbit at 6670 km (about 290 km up), sampled every 10 s for one orbit, in ITRF axes.
    seconds = np.arange(0, 5400, 10.0)
    angle = 2 * np.pi * seconds / 5440.0
    radius, speed = 6.67e6, 7.73e3
    positions = radius * np.column_stack([np.cos(angle), np.zeros_like(angle), np.sin(angle)])
    velocities = speed * np.column_stack([-np.sin(angle), np.zeros_like(angle), np.cos(angle)])
    start = np.datetime64("2023-12-18T09:52:13", "ns")
    times = start + (seconds * 1e9).astype("timedelta64[ns]")
    states = OrbitStates(times, "GPS", positions, velocities, "ITRF", "circular test orbit")
    attitude = attitude_from_law(states, along_velocity="+x", towards_nadir="+z")

print(
    f"{len(states)} epochs, {str(states.times[0])[:19]} to {str(states.times[-1])[11:19]} {states.time_scale}"
)
print("attitude:", attitude.source)

Write the relative velocity in the body frame, and describe its direction by two angles: the **azimuth** in the body's x–y plane, measured from +x towards +y, and the **elevation** out of that plane, towards +z (nadir). An azimuth near 0° means the satellite flies +x first, near ±180° means −x first. The air hits the faces on that side.

In [39]:
import plotly.graph_objects as go

velocity_in_body = attitude.body_from_reference.apply(states.velocities_m_per_s)
direction = velocity_in_body / np.linalg.norm(velocity_in_body, axis=1)[:, np.newaxis]
azimuth_deg = np.degrees(np.arctan2(direction[:, 1], direction[:, 0]))
elevation_deg = np.degrees(np.arcsin(direction[:, 2]))
print(
    f"azimuth from +x: median {np.median(azimuth_deg):.1f}°, "
    f"from {azimuth_deg.min():.1f}° to {azimuth_deg.max():.1f}°"
)
print(
    f"elevation towards +z: median {np.median(elevation_deg):.2f}°, "
    f"from {elevation_deg.min():.2f}° to {elevation_deg.max():.2f}°"
)

figure = go.Figure()
figure.add_scatter(x=states.times, y=azimuth_deg, mode="markers", marker_size=3, name="azimuth from +x")
figure.add_scatter(
    x=states.times, y=elevation_deg, mode="markers", marker_size=3, name="elevation towards +z"
)
figure.update_layout(
    title="Direction of the velocity relative to the air, in the body frame",
    yaxis_title="degrees",
    xaxis_title=f"time ({states.time_scale})",
    height=380,
)
figure

azimuth from +x: median 0.0°, from -3.6° to 3.6°
elevation towards +z: median 0.00°, from -0.00° to 0.00°


With FM134's files (2023-12-18), the 493 shared observed seconds give an azimuth near −162°: the satellite flies **−x first**, turned about 18° in yaw. The elevation stays within about a degree of the x–y plane, so the body's nadir axis really does point down. That matches the attitude file's own story: the yaw sat near 180° for most of those two hours.

With the circular orbit and the law, the azimuth swings between about −3.6° and +3.6° around the orbit, not exactly 0°. The law points +x along the **inertial** flight direction, and the air's direction differs from it by the Earth's rotation: most at the equator, nothing over the poles.

## 13. The flow on each panel (A4b)

For every panel at every epoch, `panel_flow_geometry` gives what the gas-surface models need:

- **cos δ:** how squarely the flow meets the panel. +1 is head-on (ram), 0 edge-on (the flow skims along it), negative facing away (wake).
- **t̂:** the direction the shear acts along the panel.

`plot_flow_on_spacecraft` colours each panel by cos δ, blue (−1) through white (0) to red (+1), and draws the flow as an arrow coming in.

The spacecraft here is the **3U CubeSat example** (section 9), flown with the attitude and orbit of section 12: FM134's measured attitude if your files were found, otherwise the attitude law. **It is not FM134's geometry**, which isn't in the project. It shows what leodragkit does with a real attitude.

In [ ]:
from leodragkit.aero.frames import panel_flow_geometry
from leodragkit.io.geometry_files import load_spacecraft_description
from leodragkit.plots.spacecraft import plot_flow_on_spacecraft

example_path = Path("..") / "config" / "geometry" / "cubesat_3u_example.yaml"
cubesat_3u = build_spacecraft(load_spacecraft_description(example_path), source=example_path.name)
flow_geometry = panel_flow_geometry(cubesat_3u.panel_set, attitude, states)
print(f"{flow_geometry.panel_count} panels × {len(flow_geometry)} epochs")
plot_flow_on_spacecraft(cubesat_3u.panel_set, flow_geometry, epoch_index=0)

A first number from cos δ: the **area facing the flow**, Σ A·cos δ over the panels with cos δ > 0. For a closed convex body this is its silhouette seen from upstream: the cross-section that matters most for drag. The two-sided solar panels count once, whichever side faces the flow.

In [ ]:
facing = np.clip(flow_geometry.cos_incidence, 0, None) * cubesat_3u.panel_set.areas_m2  # (T, N)
area_facing_m2 = facing.sum(axis=1)
by_component = {
    name: facing[:, cubesat_3u.panel_set.component_ids == index].sum(axis=1)
    for index, name in enumerate(cubesat_3u.panel_set.component_names)
}
print(
    f"area facing the flow: median {np.median(area_facing_m2):.4f} m², "
    f"from {area_facing_m2.min():.4f} to {area_facing_m2.max():.4f} m²"
)
for name, area in by_component.items():
    print(f"  {name:15s} median {np.median(area):.4f} m²")

With FM134's attitude (−x first, turned about 18° in yaw), the 3U faces the flow with about 0.102 m²:

- **the bus:** 0.043 m². That's its −x face, 0.1 × 0.34 m at cos 18° (0.032 m²), plus the −y side it turned towards (0.011 m²);
- **each solar panel:** 0.029 m², its −x side at cos 18°;
- **the antenna:** 0.002 m².

So the two deployed panels make up more than half the area facing the flow, which is why panels so often dominate a CubeSat's drag. Under the attitude law, +x leads within about 3.6°: the bus's +x face (0.034 m²) and the panels' +x sides (0.030 m² each) take the flow, 0.097 m² in all.

These are geometry only. The force on each panel comes next: the gas-surface coefficients (A1) at these angles, summed over the panels (A5).

## Saving a figure

Plot functions never write files. To keep a figure, save it yourself, for example into the output directory from your main config:

```python
figure.write_html("cube.html")
```

## Coming next

- **A4**: the incoming flow drawn on the spacecraft (plot level 2).